In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'    # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [5]:
reviews = pd.read_csv('../../../data/processed/steam_indie_reviews.csv')

print(f"reviews: {reviews.shape}")
reviews.head(5)

reviews: (150285, 21)


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,steam_purchase,received_for_free,written_during_early_access,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played
0,137795963,1432860,english,"The way there are lots of buggs in this game, ...",1683231392,1683231392,False,9,0,0.571289,...,True,False,False,76561198093493589,0,32,1360,0,937,1684544586
1,137795273,1432860,english,After decades playing farm games (Harvest Moo...,1683230308,1683230308,True,1,0,0.523810,...,True,False,False,76561198013915791,512,14,9986,0,2254,1721609985
2,137794614,1432860,french,Rempli de bugs au point que je ne peux même pl...,1683229419,1683229419,False,1,0,0.473322,...,True,False,False,76561198032695961,0,1,267,0,267,1683229271
3,137794475,1432860,english,The game should have launched fully complete a...,1683229182,1683229460,False,5,1,0.527390,...,True,False,False,76561198271137038,955,6,3721,0,3395,1731850878
4,137793664,1432860,french,Impossible à l'heure actuelle de recommander l...,1683228015,1689689734,False,1,0,0.492002,...,True,False,False,76561198412568279,0,1,1021,0,953,1731696034


In [6]:

import pandas as pd

games = pd.read_csv('../../../data/processed/steam_stratified_74_games.csv')
games['release_date'] = pd.to_datetime(games['release_date'])
games['cutoff_date'] = games['release_date'] + pd.Timedelta(days=90)

reviews['review_date'] = pd.to_datetime(reviews['timestamp_created'], unit='s')

agg = reviews.groupby('appid')['review_date'].agg(
    review_count='count',
    first_review='min',
    last_review='max'
).reset_index()

result = games[['appid', 'name_store', 'release_date', 'cutoff_date']].merge(agg, on='appid', how='left')
result['days_covered'] = (result['last_review'] - result['release_date']).dt.days

# 수집 스크립트가 90일 이내 리뷰만 가져오므로 last_review는 최대 day ~89
# → cutoff_date(90일) 기준 7일 이내까지 수집됐으면 3개월 데이터 보유로 판단
result['has_3m_data'] = (
    result['review_count'].notna() &
    (result['last_review'] >= result['cutoff_date'] - pd.Timedelta(days=7))
)

print(f"전체 게임 수: {len(result)}")
print(f"3개월 데이터 보유: {result['has_3m_data'].sum()}개")
print(f"3개월 데이터 미보유: {(~result['has_3m_data']).sum()}개\n")

missing = result[~result['has_3m_data']][
    ['appid', 'name_store', 'release_date', 'cutoff_date', 'review_count', 'last_review', 'days_covered']
]
if missing.empty:
    print("모든 게임이 출시 후 3개월(90일) 데이터를 보유하고 있습니다.")
else:
    print("3개월 데이터 미보유 게임:")
    print(missing.to_string(index=False))


전체 게임 수: 74
3개월 데이터 보유: 68개
3개월 데이터 미보유: 6개

3개월 데이터 미보유 게임:
  appid               name_store release_date cutoff_date  review_count         last_review  days_covered
2334220 Home Sweet Home : Online   2023-06-21  2023-09-19           NaN                 NaT           NaN
2379780                  Balatro   2024-02-20  2024-05-20           NaN                 NaT           NaN
3393750                     我的人生   2025-03-05  2025-06-03         390.0 2025-05-20 11:22:08          76.0
2715370      Ancient Cultivatrix   2025-02-13  2025-05-14          64.0 2025-05-04 15:52:17          80.0
1106140      The Drift Challenge   2023-06-29  2023-09-27           6.0 2023-09-04 17:15:10          67.0
1239300           Gravewood High   2023-05-03  2023-08-01           3.0 2023-05-14 19:16:11          11.0
